# Progetto 5G Networks (Gruppo 2)
## Sorgente da trasmettere

In [52]:
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PERCORSO = Path("dataset") / "entropy.txt"
if not PERCORSO.exists():
    PERCORSO = Path("notebooks") / "dataset" / "entropy.txt"

text = PERCORSO.read_text(encoding="utf-8")
dati = text.encode("utf-8")

print("LUNGHEZZA TESTO:\t", len(text), "caratteri")
print("DIMENSIONE IN BIT:\t", len(dati) * 8, "bit\n")
print(text[:10000])

LUNGHEZZA TESTO:	 1842 caratteri
DIMENSIONE IN BIT:	 14736 bit

A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.
   The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual 

## Codifica di sorgente: LZ78

L'header (numero di coppie, 32 bit) viene ripetuto `REP` volte e decodificato a maggioranza, così un singolo bit errato non lo corrompe. Il decoder si ferma quando i bit finiscono.

In [53]:
REP = 5  # ripetizioni dell'header

def lz78_encode(data: bytes) -> np.ndarray:
    dizionario, coppie, corrente = {}, [], b""
    for b in data:
        c = bytes([b])
        if corrente + c in dizionario:
            corrente += c
        else:
            coppie.append((dizionario.get(corrente, 0), b))
            dizionario[corrente + c] = len(dizionario) + 1
            corrente = b""
    if corrente:
        coppie.append((dizionario.get(corrente[:-1], 0), corrente[-1]))
    return np.array(coppie, dtype=np.int64)


def lz78_decode(coppie: np.ndarray) -> bytes:
    dizionario = [b""]
    out = bytearray()
    for indice, simbolo in coppie:
        prefisso = dizionario[indice] if indice < len(dizionario) else b""  # indice corrotto -> fallback
        s = prefisso + bytes([int(simbolo)])
        out += s
        dizionario.append(s)
    return bytes(out)


def _int_to_bits(x: int, w: int) -> np.ndarray:
    return ((x >> np.arange(w - 1, -1, -1)) & 1).astype(np.int8)


def _bits_to_int(b: np.ndarray) -> int:
    return int(np.dot(b.astype(np.int64), 1 << np.arange(len(b) - 1, -1, -1, dtype=np.int64))) if len(b) else 0


def coppie_to_bits(coppie: np.ndarray) -> np.ndarray:
    n = len(coppie)
    pezzi = [np.tile(_int_to_bits(n, 32), REP)]
    for i, (indice, simbolo) in enumerate(coppie):
        w = i.bit_length()
        if w:
            pezzi.append(_int_to_bits(int(indice), w))
        pezzi.append(_int_to_bits(int(simbolo), 8))
    return np.concatenate(pezzi).astype(np.int8)


def bits_to_coppie(bits: np.ndarray) -> np.ndarray:
    hdr = bits[:32 * REP].reshape(REP, 32)
    n = _bits_to_int((hdr.sum(axis=0) * 2 > REP).astype(np.int64))  # voto a maggioranza
    pos, coppie = 32 * REP, []
    for i in range(n):
        w = i.bit_length()
        if pos + w + 8 > len(bits):      # bit finiti (stream troncato / n errato)
            break
        indice = _bits_to_int(bits[pos:pos + w]); pos += w
        simbolo = _bits_to_int(bits[pos:pos + 8]); pos += 8
        coppie.append((indice, simbolo))
    return np.array(coppie, dtype=np.int64).reshape(-1, 2)

In [54]:
compresso = lz78_encode(dati)
bits = coppie_to_bits(compresso)
print(len(compresso), "coppie ->", len(bits), "bit\n")

print(*compresso)

644 coppie -> 10729 bit

[ 0 65] [ 0 32] [ 0 77] [ 0 97] [  0 116] [  0 104] [  0 101] [  0 109] [  4 116] [  0 105] [ 0 99] [  4 108] [ 2 84] [  6 101] [  0 111] [  0 114] [  0 121] [  2 111] [  0 102] [ 2 67] [ 15 109] [  8 117] [  0 110] [10 99] [  9 105] [ 15 110] [ 0 10] [ 0 66] [17 32] [ 0 67] [ 0 46] [ 2 69] [31 32] [ 0 83] [ 0 72] [ 1 78] [ 0 78] [ 0 79] [37 10] [ 0 73] [37 84] [ 0 82] [38 68] [ 0 85] [30 84] [40 79] [39 84] [35 69] [  2 114] [ 7 99] [  7 110] [ 5 32] [  0 100] [  7 118] [  7 108] [ 15 112] [  8 101] [ 23 116] [ 18 102] [  2 118] [  4 114] [ 10 111] [  0 117] [  0 115] [  2 109] [  7 116] [  6 111] [ 53 115] [59 32] [  8 111] [ 53 117] [  0 108] [ 25 111] [23 32] [ 64 117] [ 11 104] [ 2 97] [64 32] [ 0 80] [30 77] [ 77 110] [53 32] [79 80] [ 3 32] [  0 119] [  6 105] [76 32] [  7 120] [76 97] [ 23 103] [ 7 32] [ 0 98] [  4 110] [ 53 119] [ 10 100] [  5 104] [  2 102] [ 15 114] [  2 115] [ 10 103] [23 97] [72 45] [  5 111] [ 0 45] [ 23 111] [ 10 115] [ 91 114] [

In [55]:
print(*bits)

0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1 1 0 1 0 0 0 1 1 0 0 0 0 1 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 0 1 1 0 1 1 0 1 0 1 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 1 0 0 0 0 0 1 1 0 0 0 1 1 0 1 0 0 0 1 1 0 1 1 0 0 0 0 1 0 0 1 0 1 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 1 0 0 0 0 0 1 1 0 1 1 1 1 0 0 0 0 0 1 1 1 0 0 1 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0 0 0 1 0 0 1 1 0 1 1 1 1 0 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 1 0 0 1 0 0 0 0 1 1 0 1 1 1 1 0 1 1 0 1 1 0 1 0 1 0 0 0 0 1 1 1 0 1 0 1 0 0 0 0 0 0 1 1 0 1 1 1 0 0 1 0 1 0 0 1 1 0 0 0 1 1 0 1 0 0 1 0 1 1 0 1 0 0 1 0 1 1 1 1 0 1 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 1 0 0 

## Canale: 2-PAM + AWGN + fading

In [56]:
def sendPAM(SNR_dB, sorgente, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    simboli = 2 * sorgente.astype(int) - 1
    SNR = 10 ** (SNR_dB / 10)
    sigma = np.sqrt(1 / (2 * SNR))
    rumore = sigma * rng.standard_normal(len(sorgente))
    a = np.sqrt(rng.exponential(1.0, len(sorgente)))   # E esponenziale media 1
    return ((a * simboli + rumore) > 0).astype(np.int8)


def ber_teorica(SNR_dB):
    g = 10 ** (np.asarray(SNR_dB) / 10)
    return 0.5 * (1 - np.sqrt(g / (1 + g)))

## Ricezione e statistiche

In [57]:
from rapidfuzz.distance import Levenshtein

def trasmetti(SNR_dB, seed=None):
    rng = np.random.default_rng(seed)
    rx = sendPAM(SNR_dB, bits, rng)
    ricevuto = lz78_decode(bits_to_coppie(rx)).decode("utf-8", errors="replace")
    errori_bit = int(np.sum(rx != bits))
    cer = Levenshtein.distance(text, ricevuto) / len(text)
    return rx, ricevuto, errori_bit, cer

In [58]:
for i in range(11):
    SNR_dB = i * 4
    rx, ricevuto, errori_bit, cer = trasmetti(SNR_dB, seed=1)
    bit_orig, bit_comp = len(dati) * 8, len(bits)
    print(f"\n=== STATISTICHE (SNR_dB: {SNR_dB} dB) ===")
    print(f"BER:  {errori_bit/len(bits):.3e} ({errori_bit} errati su {len(bits)}) | teorica {ber_teorica(SNR_dB):.3e}")
    print(f"CER (Levenshtein): {cer:.2%}")
    print(f"Testo identico: {text == ricevuto}")
    print(f"Compressione: {bit_orig/bit_comp:.2f}:1 | spazio risparmiato {100*(1-bit_comp/bit_orig):.2f}%\n")
    print(ricevuto[:10000])


=== STATISTICHE (SNR_dB: 0 dB) ===
BER:  1.457e-01 (1563 errati su 10729) | teorica 1.464e-01
CER (Levenshtein): 88.71%
Testo identico: False
Compressione: 1.37:1 | spazio risparmiato 27.19%

� �a�`em�4i+ l D`e/rr9 of C/-id on/c/y/n
�br�Gm�aid0sJ�Ns\�O
a�R�NDOGpOM
DGR�' ofa!d/�
l
Dpi� on4/n�ivari��n t�t`odP�ND/c}i�u�n
� on �tY+� o�� Sm�M+nidxSQ�q�nwG� D SQ8� � on�e dPraOdU+f�* f`c`c�ReqS,s\?˭ onm l3e r�4*`ASm on4af�JO+es|`e"R�er9u� | h l3q�tg l3qq/cq�n��*e x�nw�/-mn�ya!.A ba!.w l3qfo e rga8dP �r9yv c/nt�� D � on�jOM}`e"cm1�OtaO0 xP l0A2� Gr9�Jy xii g/nIivt t�i��OMm1a�*e`j�'tJIJymbc/n�taO0a�pm�!�qE��inive�ynidxpe x`e�r9y`s\?$RB tua!�
�m�q�4CAoaO0a^e�JOanpsG x, l3qnaO0abytarf�*e`e�/nenpyf n`emidxq"��#qoAl x, g aO0ab3�VRg� p/q�n(dPrd
de xe l3qu/ aO0ab3tr�'g a l3qqsar/n�c�u
l`f�fkaa�'g+n��*eic��*g on�i�t/c}	�sGldP[ Duor9��`e"ve�j/c}	-� |is\|�n�o`AVliR bay{ on�
`�/cau�tq hns\h�4*x bao t�e,f�c/-m5ei on�`y{n G`a!�r+tds|{aO0a^"m`/nmivgd5 %�NDt`ebivgxac�NDhi��� lz� Gt/x